# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

In [ ]:
%load_ext autoreload
%autoreload 2    

In [ ]:
import sys
from pathlib import Path

course_root = Path.cwd().parent
# while not (course_root / "tools").exists():
#    course_root = course_root.parent

sys.path.insert(0, str(course_root) + "/part1")
print("course_root:", course_root)

import json
import random

import numpy as np
import pandas as pd
from data import GoEmotions

ds = GoEmotions(drop_multi_labels=True, merge_train_dev=True)
ds.summary

In [ ]:
sys.path.insert(0, str(course_root))
import torch
from part2.clslib import (
    PretrainedEmbeddings,
    collate_sequences,
    EmbedAndPoolEncoder,
    SoftmaxRegressor,
    Tokenizer,
)

emb = PretrainedEmbeddings(
    path="wiki-news-300d-1M.vec.zip",  # update path if needed
    num_dims=300,
    skip_first_line=True,
)

tok = Tokenizer(
    vocab=emb.vocab, tokenizer_path=str(course_root / "tools" / "tokenizer.perl")
)

print("Vocab size:", len(emb.vocab))
print("Embedding matrix shape:", emb.vectors.shape)

In [ ]:
out = tok.tokens(ds.train["text"].tolist())
print("Sample tokenized sentences:", out[:5])

train_ids = [
    [emb.index(t) for t in toks] for toks in tok.tokens(ds.train["text"].tolist())
]
print("Sample IDs:", train_ids[:5])

In [ ]:
embedding_matrix = torch.from_numpy(emb.vectors)

encoder = EmbedAndPoolEncoder(
    embedding_matrix=embedding_matrix,
    average_pool=True,
    padding_idx=emb.vocab.index(emb.pad_token),
    freeze_embeddings=True,
)

num_classes = ds.train["label"].nunique()  # adjust column name if different
model = SoftmaxRegressor(
    hidden_size1=128,
    hidden_size2=64,
    num_classes=num_classes,
    encoder=encoder,
    dropout=0.2,
)

# sanity check on a mini-batch of 4 examples
batch_x = collate_sequences(train_ids[:4], padding_value=encoder.pad_idx)
log_probs = model(batch_x)
print("Batch shape:", batch_x.shape, "-> Log-probs shape:", log_probs.shape)

In [ ]:
from part2.clslib import LSTMEncoder

lstm_enc = LSTMEncoder(
    hidden_size=2,
    embedding_matrix=embedding_matrix,
    padding_idx=emb.vocab.index(emb.pad_token),
    freeze_embeddings=True,
)

In [ ]:
lstm_enc(batch_x)

In [ ]:
import part1
import part2
from collections import OrderedDict

In [ ]:
my_seeds = [0, 42, 99]
num_folds = 3
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(ds.train["text"], ds.train["label"])), k=num_folds, seed=seed
        )
    )

emotions_outcome_space = part1.clslib.NamedClasses(names=ds.emotions)

In [ ]:
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)
sweep_csv = results_folder / "sweep.csv"

shared_opt = {"weight_decay": 1e-4, "batch_size": 256, "num_epochs": 300, "patience": 5}

sweeps = {
    "embed-and-pool": {
        "base": {
            "enc_type": "embed-and-pool",
            "enc_kwargs": {"freeze_embeddings": True},
            "regressor_kwargs": {},
            "opt_kwargs": dict(shared_opt),
        },
        "search_space": {
            "regressor_kwargs.hidden_size1": ("choice", [100, 300, 600]),
            "regressor_kwargs.hidden_size2": ("choice", [50, 100]),
            "regressor_kwargs.dropout": ("uniform", 0.0, 0.5),
            "opt_kwargs.learning_rate": ("loguniform", 3e-4, 3e-2),
        },
        "N": 40,
    },
    "lstm": {
        "base": {
            "enc_type": "lstm",
            "enc_kwargs": {"freeze_embeddings": True},
            "regressor_kwargs": {"hidden_size1": 50, "hidden_size2": 50},
            "opt_kwargs": dict(shared_opt),
        },
        "search_space": {
            "enc_kwargs.hidden_size": ("choice", [20, 50, 100]),
            "enc_kwargs.word_dropout": ("uniform", 0.0, 0.3),
            "regressor_kwargs.dropout": ("uniform", 0.0, 0.5),
            "opt_kwargs.learning_rate": ("loguniform", 3e-4, 3e-2),
        },
        "N": 30,
    },
}


def sample(spec, rng):
    kind = spec[0]
    if kind == "choice":
        return rng.choice(spec[1])
    if kind == "uniform":
        return round(rng.uniform(spec[1], spec[2]), 2)
    if kind == "loguniform":
        return float(f"{10 ** rng.uniform(np.log10(spec[1]), np.log10(spec[2])):.2g}")
    raise ValueError(kind)


def make_config(base, params):
    cfg = json.loads(json.dumps(base))
    for key, value in params.items():
        group, name = key.split(".")
        cfg[group][name] = value
    return cfg


def config_name(enc_type, params):
    return (
        enc_type + "_" + "_".join(f"{k.split('.')[1]}={v}" for k, v in params.items())
    )


done = set(pd.read_csv(sweep_csv)["name"]) if sweep_csv.exists() else set()
seed, run = my_seeds[0], 0

for enc_type, sweep in sweeps.items():
    rng = random.Random(42)
    for _ in range(sweep["N"]):
        params = {k: sample(spec, rng) for k, spec in sweep["search_space"].items()}
        name = config_name(enc_type, params)
        if name in done:
            continue
        cfg = make_config(sweep["base"], params)
        result = part2.experiment.cross_validation(
            data_splits=my_cross_validation_sets[run],
            outcome_space=emotions_outcome_space,
            tokenizer=tok,
            emb_model=emb,
            device="cuda",
            save_to=f"{results_folder}/sweep/{name}/seed-{seed}",
            **cfg,
        )
        accs = list(result["accuracies"].values())
        row = {
            "name": name,
            "enc_type": enc_type,
            **cfg["enc_kwargs"],
            **cfg["regressor_kwargs"],
            **cfg["opt_kwargs"],
            "mean_acc": np.mean(accs),
            "std_acc": np.std(accs),
        }
        df = pd.read_csv(sweep_csv) if sweep_csv.exists() else pd.DataFrame()
        pd.concat([df, pd.DataFrame([row])]).to_csv(sweep_csv, index=False)

res = pd.read_csv(sweep_csv)
res.sort_values("mean_acc", ascending=False).groupby("enc_type").head(5)

In [ ]:
TRAIN = False

final_seeds = [0, 42, 99]
final_cross_validation_sets = []

for seed in final_seeds:
    final_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(ds.train["text"], ds.train["label"])), k=5, seed=seed
        )
    )

configs = {
    "embed-and-pool": {
        "enc_type": "embed-and-pool",
        "enc_kwargs": {
            "freeze_embeddings": True,
        },
        "regressor_kwargs": {
            "hidden_size1": 600,
            "hidden_size2": 100,
            "dropout": 0.07,
        },
        "opt_kwargs": {
            "learning_rate": 5.7e-4,
            "weight_decay": 1e-4,
            "batch_size": 256,
            "num_epochs": 300,
            "patience": 5,
        },
    },
    "lstm": {
        "enc_type": "lstm",
        "enc_kwargs": {
            "freeze_embeddings": True,
            "hidden_size": 100,
            "word_dropout": 0.21,
        },
        "regressor_kwargs": {
            "hidden_size1": 50,
            "hidden_size2": 50,
            "dropout": 0.21,
        },
        "opt_kwargs": {
            "learning_rate": 2.4e-3,
            "weight_decay": 1e-4,
            "batch_size": 256,
            "num_epochs": 300,
            "patience": 5,
        },
    },
}

my_classifiers = OrderedDict()
for config in configs:
    my_classifiers[config] = []
    for run, seed in enumerate(final_seeds):
        run_folder = f"{results_folder}/{config}/seed-{seed}"
        if TRAIN or not Path(run_folder).exists():
            result = part2.experiment.cross_validation(
                data_splits=final_cross_validation_sets[run],
                outcome_space=emotions_outcome_space,
                tokenizer=tok,
                emb_model=emb,
                enc_type=configs[config]["enc_type"],
                enc_kwargs=configs[config]["enc_kwargs"],
                regressor_kwargs=configs[config]["regressor_kwargs"],
                opt_kwargs=configs[config]["opt_kwargs"],
                device="cuda",  #'cpu'
                save_to=run_folder,
            )
        else:
            result = part2.experiment.load_cross_validation_experiment(
                data_splits=final_cross_validation_sets[run],
                tokenizer=tok,
                emb_model=emb,
                load_from=run_folder,
                device="cuda",
            )
        my_classifiers[config].append(result)